# 03 — Injetar erros e validar

Objetivo: criar tentativas sintéticas com erro controlado e soluções sem erro. Entrada: `data/interim/problemas_localizados.jsonl`. Saída: `data/interim/erros_validados.jsonl`. As contas são avaliadas por um interpretador restrito baseado em AST; nenhuma expressão é passada a `eval`. Casos incompatíveis com a validação são descartados e contabilizados.

In [1]:
# Opcional: %pip install sympy pydantic>=2.7
import ast
import json
import random
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Any, Literal

from pydantic import BaseModel, ConfigDict

def localizar_projeto() -> Path:
    '''Encontra a pasta de dados sem depender do kernel anterior.'''
    for base in (Path.cwd(), *Path.cwd().parents):
        pasta = base / 'notebooks' / 'eja_diagnosticador_dataset'
        if (pasta / 'data' / 'config.json').exists(): return pasta
        if base.name == 'eja_diagnosticador_dataset' and (base / 'data' / 'config.json').exists(): return base
    raise FileNotFoundError('Configuração compartilhada não encontrada')

ROOT = localizar_projeto()
CONFIG = json.loads((ROOT / 'data' / 'config.json').read_text(encoding='utf-8'))
random.seed(CONFIG['seed'])
ENTRADA = ROOT / 'data' / 'interim' / 'problemas_localizados.jsonl'
SAIDA = ROOT / 'data' / 'interim' / 'erros_validados.jsonl'
TIPOS = list(CONFIG['error_proportions'])

class RegistroErro(BaseModel):
    '''Esquema de uma tentativa sintética antes de gravar JSONL.'''
    model_config = ConfigDict(extra='allow')
    id: str
    base_id: str
    erro_tipo: Literal['ERRO_OPERACAO', 'ERRO_CALCULO', 'PASSO_OMITIDO', 'ERRO_LEITURA', 'SEM_ERRO']
    tentativa: dict[str, Any]

def serializar_jsonl(registro: dict[str, Any]) -> str:
    '''Escapa separadores Unicode para manter um registro por linha física.'''
    texto = json.dumps(registro, ensure_ascii=False)
    for caractere, escape in ((chr(0x85), r'\u0085'), (chr(0x2028), r'\u2028'), (chr(0x2029), r'\u2029')):
        texto = texto.replace(caractere, escape)
    return texto

def calcular_no_seguro(expressao: str) -> Decimal:
    '''Avalia somente números e operações aritméticas permitidas na AST.'''
    arvore = ast.parse(expressao.replace('^', '**'), mode='eval').body
    def visitar(no: ast.AST) -> Decimal:
        if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
            return Decimal(str(no.value))
        if isinstance(no, ast.UnaryOp) and isinstance(no.op, (ast.UAdd, ast.USub)):
            valor = visitar(no.operand)
            return valor if isinstance(no.op, ast.UAdd) else -valor
        if isinstance(no, ast.BinOp):
            a, b = visitar(no.left), visitar(no.right)
            if isinstance(no.op, ast.Add): return a + b
            if isinstance(no.op, ast.Sub): return a - b
            if isinstance(no.op, ast.Mult): return a * b
            if isinstance(no.op, ast.Div):
                if b == 0: raise ValueError('Divisão por zero')
                return a / b
            if isinstance(no.op, ast.Pow) and b == int(b) and abs(b) <= 6: return a ** int(b)
        raise ValueError(f'Expressão não permitida: {ast.dump(no)}')
    return visitar(arvore)

def valor(texto: str) -> Decimal:
    '''Converte um resultado anotado para Decimal.'''
    texto = texto.strip()
    if re.fullmatch(r'[-+]?\d{1,3}(,\d{3})+(\.\d+)?', texto):
        texto = texto.replace(',', '')
    elif re.fullmatch(r'[-+]?\d{1,3}(\.\d{3})+(,\d+)?', texto):
        texto = texto.replace('.', '').replace(',', '.')
    else:
        texto = texto.replace(',', '.')
    try: return Decimal(texto)
    except InvalidOperation as erro: raise ValueError(f'Resultado não numérico: {texto}') from erro

def primeira_conta_valida(registro: dict[str, Any]) -> tuple[str, Decimal] | None:
    '''Seleciona uma conta anotada cujo resultado confere com a calculadora.'''
    for conta in registro.get('contas_anotadas', []):
        try:
            calculado = calcular_no_seguro(conta['expressao'])
            if calculado == valor(conta['resultado']): return conta['expressao'], calculado
        except (ValueError, SyntaxError, InvalidOperation):
            continue
    return None

def trocar_operacao(expressao: str) -> tuple[str, Decimal] | None:
    '''Troca uma operação binária e retorna a nova conta quando muda o resultado.'''
    arvore = ast.parse(expressao, mode='eval')
    correto = calcular_no_seguro(expressao)
    operadores = [(ast.Add, ast.Sub), (ast.Sub, ast.Add), (ast.Mult, ast.Add), (ast.Div, ast.Mult)]
    for no in ast.walk(arvore):
        if isinstance(no, ast.BinOp):
            for original, novo in operadores:
                if isinstance(no.op, original):
                    anterior = no.op
                    no.op = novo()
                    alterada = ast.unparse(arvore)
                    try: resultado = calcular_no_seguro(alterada)
                    except (ValueError, SyntaxError, InvalidOperation):
                        no.op = anterior
                        continue
                    if resultado != correto: return alterada, resultado
                    no.op = anterior
    return None

def trocar_operando(expressao: str) -> tuple[str, Decimal] | None:
    '''Substitui um dado numérico por outro para simular erro de leitura.'''
    arvore = ast.parse(expressao, mode='eval')
    for no in ast.walk(arvore):
        if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
            original = no.value
            no.value = original + 1
            alterada = ast.unparse(arvore)
            try: resultado = calcular_no_seguro(alterada)
            except (ValueError, SyntaxError, InvalidOperation):
                no.value = original
                continue
            if resultado != calcular_no_seguro(expressao): return alterada, resultado
            no.value = original
    return None

def construir_exemplo(registro: dict[str, Any], tipo: str) -> dict[str, Any] | None:
    '''Gera uma tentativa e exige evidência de erro ou correção para o rótulo.'''
    conta = primeira_conta_valida(registro)
    if conta is None: return None
    expressao, correto = conta
    passos = list(registro.get('passos_raciocinio', []))
    tentativa: dict[str, Any] = {'expressao_original': expressao, 'resultado_correto': str(correto)}
    if tipo == 'ERRO_OPERACAO':
        alterada = trocar_operacao(expressao)
        if alterada is None: return None
        tentativa.update(expressao_tentada=alterada[0], resultado_tentado=str(alterada[1]))
        assert alterada[1] != correto
    elif tipo == 'ERRO_CALCULO':
        errado = correto + 1
        tentativa.update(expressao_tentada=expressao, resultado_tentado=str(errado))
        assert errado != calcular_no_seguro(expressao)
    elif tipo == 'ERRO_LEITURA':
        alterada = trocar_operando(expressao)
        if alterada is None: return None
        tentativa.update(expressao_tentada=alterada[0], resultado_tentado=str(alterada[1]))
        assert alterada[1] != correto
    elif tipo == 'PASSO_OMITIDO':
        if len(passos) < 2: return None
        tentativa['passos_tentados'] = passos[:-1]
        tentativa['passo_omitido'] = passos[-1]
        assert len(tentativa['passos_tentados']) < len(passos)
    elif tipo == 'SEM_ERRO':
        contas = registro.get('contas_anotadas', [])
        conferidas = []
        for item in contas:
            calculado = calcular_no_seguro(item['expressao'])
            if calculado != valor(item['resultado']): return None
            conferidas.append((item['expressao'], calculado))
        resposta_final = Decimal(registro['resposta_numerica'])
        correspondentes = [item for item in conferidas if item[1] == resposta_final]
        if not correspondentes: return None
        expressao, correto = correspondentes[-1]
        tentativa.update(expressao_original=expressao, resultado_correto=str(correto))
        tentativa.update(expressao_tentada=expressao, resultado_tentado=str(correto), passos_tentados=passos)
        assert calcular_no_seguro(expressao) == correto
    else:
        raise ValueError(f'Tipo de erro desconhecido: {tipo}')
    return {**registro, 'erro_tipo': tipo, 'tentativa': tentativa}

if not ENTRADA.exists(): raise FileNotFoundError(f'Execute primeiro o notebook 02: {ENTRADA}')
localizados = [json.loads(l) for l in ENTRADA.read_text(encoding='utf-8').splitlines() if l]
proporcoes = CONFIG['error_proportions']
soma = sum(proporcoes.values())
tipos_por_indice = [tipo for tipo in TIPOS for _ in range(round(CONFIG['max_examples'] * proporcoes[tipo] / soma))]
random.shuffle(tipos_por_indice)
SAIDA.parent.mkdir(parents=True, exist_ok=True)
descartes: dict[str, int] = {tipo: 0 for tipo in TIPOS}
gerados: list[dict[str, Any]] = []
for indice, registro in enumerate(localizados):
    tipo = tipos_por_indice[indice % len(tipos_por_indice)] if tipos_por_indice else 'SEM_ERRO'
    try:
        exemplo = construir_exemplo(registro, tipo)
        if exemplo is None: descartes[tipo] += 1
        else: gerados.append(RegistroErro.model_validate(exemplo).model_dump())
    except (ValueError, AssertionError, SyntaxError, InvalidOperation) as erro:
        descartes[tipo] += 1
        print(f"Descartado {registro.get('id')} ({tipo}): {erro}")
with SAIDA.open('w', encoding='utf-8') as arquivo:
    for exemplo in gerados: arquivo.write(serializar_jsonl(exemplo) + '\n')
print('Gerados:', len(gerados), '| descartados por rótulo:', descartes)

Gerados: 99 | descartados por rótulo: {'ERRO_OPERACAO': 0, 'ERRO_CALCULO': 0, 'PASSO_OMITIDO': 1, 'ERRO_LEITURA': 0, 'SEM_ERRO': 0}


A proporção é escolhida antes da validação; descartes podem desequilibrar os rótulos e precisam ser observados. Uma conta isolada validada não prova que todo o raciocínio original está correto. Para `SEM_ERRO`, as contas disponíveis são conferidas individualmente e professores devem revisar a solução completa.

In [2]:
validados = [json.loads(l) for l in SAIDA.read_text(encoding='utf-8').splitlines() if l]
assert len(validados) == len(gerados)
print('Total:', len(validados), '| rótulos:', {t: sum(x['erro_tipo'] == t for x in validados) for t in TIPOS})
print('Descartados:', descartes)
for item in validados[:3]: print(json.dumps({'id': item['id'], 'erro_tipo': item['erro_tipo'], 'tentativa': item['tentativa']}, ensure_ascii=False))

Total: 99 | rótulos: {'ERRO_OPERACAO': 20, 'ERRO_CALCULO': 20, 'PASSO_OMITIDO': 19, 'ERRO_LEITURA': 20, 'SEM_ERRO': 20}
Descartados: {'ERRO_OPERACAO': 0, 'ERRO_CALCULO': 0, 'PASSO_OMITIDO': 1, 'ERRO_LEITURA': 0, 'SEM_ERRO': 0}
{"id": "train:0", "erro_tipo": "ERRO_CALCULO", "tentativa": {"expressao_original": "48/2", "resultado_correto": "24", "expressao_tentada": "48/2", "resultado_tentado": "25"}}
{"id": "train:1", "erro_tipo": "ERRO_OPERACAO", "tentativa": {"expressao_original": "12/60", "resultado_correto": "0.2", "expressao_tentada": "12 * 60", "resultado_tentado": "720"}}
{"id": "train:2", "erro_tipo": "ERRO_LEITURA", "tentativa": {"expressao_original": "100/2", "resultado_correto": "50", "expressao_tentada": "101 / 2", "resultado_tentado": "50.5"}}
